In [24]:
import polars as pl
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data" 


df = pl.read_csv(
    str(DATA_DIR / "data-raw" / "household_power_consumption.txt"), 
    separator=";",
    null_values="?"
)

df

Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
str,str,f64,f64,f64,f64,f64,f64,f64
"""16/12/2006""","""17:24:00""",4.216,0.418,234.84,18.4,0.0,1.0,17.0
"""16/12/2006""","""17:25:00""",5.36,0.436,233.63,23.0,0.0,1.0,16.0
"""16/12/2006""","""17:26:00""",5.374,0.498,233.29,23.0,0.0,2.0,17.0
"""16/12/2006""","""17:27:00""",5.388,0.502,233.74,23.0,0.0,1.0,17.0
"""16/12/2006""","""17:28:00""",3.666,0.528,235.68,15.8,0.0,1.0,17.0
…,…,…,…,…,…,…,…,…
"""26/11/2010""","""20:58:00""",0.946,0.0,240.43,4.0,0.0,0.0,0.0
"""26/11/2010""","""20:59:00""",0.944,0.0,240.0,4.0,0.0,0.0,0.0
"""26/11/2010""","""21:00:00""",0.938,0.0,239.82,3.8,0.0,0.0,0.0


In [25]:
df = df.rename(str.lower)
df


date,time,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3
str,str,f64,f64,f64,f64,f64,f64,f64
"""16/12/2006""","""17:24:00""",4.216,0.418,234.84,18.4,0.0,1.0,17.0
"""16/12/2006""","""17:25:00""",5.36,0.436,233.63,23.0,0.0,1.0,16.0
"""16/12/2006""","""17:26:00""",5.374,0.498,233.29,23.0,0.0,2.0,17.0
"""16/12/2006""","""17:27:00""",5.388,0.502,233.74,23.0,0.0,1.0,17.0
"""16/12/2006""","""17:28:00""",3.666,0.528,235.68,15.8,0.0,1.0,17.0
…,…,…,…,…,…,…,…,…
"""26/11/2010""","""20:58:00""",0.946,0.0,240.43,4.0,0.0,0.0,0.0
"""26/11/2010""","""20:59:00""",0.944,0.0,240.0,4.0,0.0,0.0,0.0
"""26/11/2010""","""21:00:00""",0.938,0.0,239.82,3.8,0.0,0.0,0.0


Uno de los problemas es que tenemos una columna en un formato que no es fecha, y la hora tambien esta en string. Lo que podemos hacer de una vez es combinarlos en el tipo de dato datetime para polars

In [26]:
print(df["date"].dtype)

String


In [27]:
df = df.with_columns(
    (pl.col("date") + " " + pl.col("time"))
    .str.to_datetime("%d/%m/%Y %H:%M:%S")
    .alias("datetime")
)
df = df.drop("date", "time")
df = df.select("datetime", *df.columns[:len(df.columns)-1])
df

datetime,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3
datetime[μs],f64,f64,f64,f64,f64,f64,f64
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.36,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
…,…,…,…,…,…,…,…
2010-11-26 20:58:00,0.946,0.0,240.43,4.0,0.0,0.0,0.0
2010-11-26 20:59:00,0.944,0.0,240.0,4.0,0.0,0.0,0.0
2010-11-26 21:00:00,0.938,0.0,239.82,3.8,0.0,0.0,0.0


In [28]:
df.filter(pl.col("voltage").is_null())

datetime,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3
datetime[μs],f64,f64,f64,f64,f64,f64,f64
2006-12-21 11:23:00,null,null,null,null,null,null,null
2006-12-21 11:24:00,null,null,null,null,null,null,null
2006-12-30 10:08:00,null,null,null,null,null,null,null
2006-12-30 10:09:00,null,null,null,null,null,null,null
2007-01-14 18:36:00,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…
2010-09-28 19:09:00,null,null,null,null,null,null,null
2010-09-28 19:10:00,null,null,null,null,null,null,null
2010-09-28 19:11:00,null,null,null,null,null,null,null


Notamos que todas las columnas con valores nulos en algun atributo de hecho tiene nulos en todos los demas, puede que los valores donde falten datos sea utiles
# importante
tal vez lo mejor sea dejarlos porque puede reflejar fenomenos 

In [29]:
df = df.drop_nulls()
df.null_count()

datetime,global_active_power,global_reactive_power,voltage,global_intensity,sub_metering_1,sub_metering_2,sub_metering_3
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0


In [30]:
df.write_csv(DATA_DIR / "data-processed" / "cleaned_data.csv")